# nb_bronze_stream_vehicle_battery

**Source :** Azure Event Hub — `vehicle-battery-live` topic  
**Sink   :** `abfss://bronze@evdatalakedev2.dfs.core.windows.net/event-stream/vehicle_battery_live/`  
**Format :** Delta (append), partitioned by `event_date`  
**Trigger:** Every 30 seconds (micro-batch streaming)

This notebook runs **continuously** as Task 1 in the Databricks streaming job.  
It reads raw binary events from Event Hub, parses the JSON body, and lands them  
in the Bronze Delta table with minimal transformation.

**Prerequisite:** Maven library installed on cluster:  
`com.microsoft.azure:azure-eventhubs-spark_2.12:2.3.22`

In [ ]:
# ── Cell 1: secrets ───────────────────────────────────────────────────────────
EH_CONN_STR    = dbutils.secrets.get(scope="ev-kv-scope", key="eventhub-vehicle-battery-listen-conn-str")
SP_CLIENT_ID   = dbutils.secrets.get(scope="ev-kv-scope", key="sp-client-id")
SP_CLIENT_SEC  = dbutils.secrets.get(scope="ev-kv-scope", key="sp-client-secret")
SP_TENANT_ID   = dbutils.secrets.get(scope="ev-kv-scope", key="sp-tenant-id")
ADLS_ACCOUNT   = dbutils.secrets.get(scope="ev-kv-scope", key="adls-account-name")

In [ ]:
# ── Cell 2: ADLS OAuth config ─────────────────────────────────────────────────
spark.conf.set(f"fs.azure.account.auth.type.{ADLS_ACCOUNT}.dfs.core.windows.net",       "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{ADLS_ACCOUNT}.dfs.core.windows.net",
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{ADLS_ACCOUNT}.dfs.core.windows.net",     SP_CLIENT_ID)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{ADLS_ACCOUNT}.dfs.core.windows.net", SP_CLIENT_SEC)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{ADLS_ACCOUNT}.dfs.core.windows.net",
               f"https://login.microsoftonline.com/{SP_TENANT_ID}/oauth2/token")
print("ADLS OAuth configured.")

In [ ]:
# ── Cell 3: paths + Event Hub config ─────────────────────────────────────────
BRONZE_PATH     = f"abfss://bronze@{ADLS_ACCOUNT}.dfs.core.windows.net/event-stream/vehicle_battery_live/"
CHECKPOINT_PATH = f"abfss://bronze@{ADLS_ACCOUNT}.dfs.core.windows.net/_checkpoints/vehicle-battery-live/"

# Encrypt connection string — required by the Spark EventHubs connector
encrypted_conn = spark.sparkContext._jvm.org.apache.spark.eventhubs.EventHubsUtils.encrypt(EH_CONN_STR)

EH_CONF = {
    "eventhubs.connectionString":    encrypted_conn,
    "eventhubs.consumerGroup":        "$Default",      # Basic tier supports only $Default
    "eventhubs.startingPosition":     '{"offset":"-1","seqNo":-1,"enqueuedTime":null,"isInclusive":true}',
    "maxEventsPerTrigger":            1000,
}

print(f"Bronze path      : {BRONZE_PATH}")
print(f"Checkpoint path  : {CHECKPOINT_PATH}")

In [ ]:
# ── Cell 4: read stream from Event Hub ───────────────────────────────────────
raw_stream = (
    spark.readStream
    .format("eventhubs")
    .options(**EH_CONF)
    .load()
)

# Event Hub delivers: body (binary), partition, offset, sequenceNumber,
# enqueuedTime, publisher, partitionKey, properties, systemProperties
print("Stream schema:")
raw_stream.printSchema()

In [ ]:
# ── Cell 5: parse JSON body ───────────────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

EVENT_SCHEMA = StructType([
    StructField("event_id",                   StringType(),  True),
    StructField("vehicle_id",                 StringType(),  True),
    StructField("session_id",                 StringType(),  True),
    StructField("station_id",                 StringType(),  True),
    StructField("charger_id",                 StringType(),  True),
    StructField("battery_pct",                DoubleType(),  True),
    StructField("charging_rate_kw",           DoubleType(),  True),
    StructField("battery_temp_c",             DoubleType(),  True),
    StructField("state_of_charge_target_pct", IntegerType(), True),
    StructField("estimated_minutes_to_full",  IntegerType(), True),
    StructField("event_ts",                   StringType(),  True),
])

parsed = (
    raw_stream
    .withColumn("body_str",  F.col("body").cast("string"))
    .withColumn("parsed",    F.from_json(F.col("body_str"), EVENT_SCHEMA))
    .select(
        F.col("parsed.*"),
        F.col("enqueuedTime").alias("eh_enqueued_time"),
        F.col("partition").alias("eh_partition"),
        F.col("offset").alias("eh_offset"),
    )
)

In [ ]:
# ── Cell 6: add Bronze metadata columns ──────────────────────────────────────
bronze_df = (
    parsed
    .withColumn("_ingestion_ts", F.current_timestamp())
    .withColumn("_source",       F.lit("eventhub/vehicle-battery-live"))
    # Flag rows where JSON parsing failed (body_str not null but event_id null)
    .withColumn("_is_corrupt",   F.col("event_id").isNull())
    # Partition column — date derived from event_ts for Hive-style partitioning
    .withColumn("event_date",    F.to_date(F.col("event_ts")))
)

print("Bronze schema:")
bronze_df.printSchema()

In [ ]:
# ── Cell 7: write stream to Bronze Delta ─────────────────────────────────────
streaming_query = (
    bronze_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .partitionBy("event_date")
    .trigger(processingTime="30 seconds")
    .start(BRONZE_PATH)
)

print(f"Stream started. Query ID: {streaming_query.id}")

In [ ]:
# ── Cell 8: keep-alive monitor loop ──────────────────────────────────────────
# This cell keeps the notebook alive in the Databricks job.
# Each tick prints progress so the job UI shows activity.
import time

while streaming_query.isActive:
    prog = streaming_query.lastProgress
    if prog:
        rows_in  = prog.get("numInputRows", 0)
        rows_sec = prog.get("inputRowsPerSecond", 0.0)
        batch    = prog.get("batchId", "?")
        print(f"[Batch {batch}] inputRows={rows_in}  rate={rows_sec:.1f} rows/s")
    time.sleep(30)

# If loop exits, the query died — surface the exception
streaming_query.awaitTermination()

In [ ]:
# ── Cell 9: optional verify (run manually, not in job) ───────────────────────
verify_df = spark.read.format("delta").load(BRONZE_PATH)
print(f"Total Bronze rows: {verify_df.count()}")
verify_df.orderBy(F.col("_ingestion_ts").desc()).limit(5).display()